# 04 — Observations: Historical Data in the Model

Verify that trips and telemetry are mapped to `observed_flow` and
`observed_inventory` in the graph model, demand is derived, and
time resolution produces `period_id`.

In [ ]:
from gbp.build.pipeline import build_model
from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig

mock = DataLoaderMock({"n_stations": 10, "n_depots": 2, "n_timestamps": 168})
loader = DataLoaderGraph(mock, GraphLoaderConfig())
raw = loader.load()
resolved = build_model(raw)
raw


In [2]:
assert raw.observed_flow is not None
print(f"observed_flow: {raw.observed_flow.shape}")
raw.observed_flow.head(8)

observed_flow: (548, 8)


,source_id,target_id,commodity_category,date,quantity,quantity_unit,modal_type,resource_id
0,station_1,station_10,working_bike,2025-01-01,4.0,bike,None,None
1,station_1,station_10,working_bike,2025-01-02,4.0,bike,None,None
2,station_1,station_10,working_bike,2025-01-03,3.0,bike,None,None
3,station_1,station_10,working_bike,2025-01-04,1.0,bike,None,None
4,station_1,station_10,working_bike,2025-01-05,1.0,bike,None,None
5,station_1,station_10,working_bike,2025-01-06,1.0,bike,None,None
6,station_1,station_10,working_bike,2025-01-07,1.0,bike,None,None
7,station_1,station_2,working_bike,2025-01-01,2.0,bike,None,None


In [3]:
assert raw.observed_inventory is not None
print(f"observed_inventory: {raw.observed_inventory.shape}")
raw.observed_inventory.head(8)

observed_inventory: (70, 5)


,facility_id,commodity_category,date,quantity,quantity_unit
0,station_1,working_bike,2025-01-01,13.291667,bike
1,station_1,working_bike,2025-01-02,15.750000,bike
2,station_1,working_bike,2025-01-03,17.416667,bike
3,station_1,working_bike,2025-01-04,17.458333,bike
4,station_1,working_bike,2025-01-05,14.083333,bike
5,station_1,working_bike,2025-01-06,15.333333,bike
6,station_1,working_bike,2025-01-07,16.250000,bike
7,station_10,working_bike,2025-01-01,33.083333,bike


In [4]:
assert raw.demand is not None
print(f"demand rows: {len(raw.demand)}")
print(f"observed_flow total quantity: {raw.observed_flow['quantity'].sum():.0f}")
print(f"demand total quantity: {raw.demand['quantity'].sum():.0f}")
raw.demand.head(8)

demand rows: 70
observed_flow total quantity: 1167
demand total quantity: 1167


,facility_id,date,commodity_category,quantity,quantity_unit
0,station_1,2025-01-01,working_bike,14.0,bike
1,station_1,2025-01-02,working_bike,16.0,bike
2,station_1,2025-01-03,working_bike,19.0,bike
3,station_1,2025-01-04,working_bike,21.0,bike
4,station_1,2025-01-05,working_bike,19.0,bike
5,station_1,2025-01-06,working_bike,12.0,bike
6,station_1,2025-01-07,working_bike,14.0,bike
7,station_10,2025-01-01,working_bike,21.0,bike


In [5]:
from gbp.build.validation import validate_raw_model

result = validate_raw_model(raw)
print(f"valid: {result.is_valid}, warnings: {result.has_warnings}")
for e in result.errors:
    if e.entity.startswith("observed"):
        print(f"  {e.level}: {e.entity} — {e.message}")

valid: True, warnings: False


In [6]:
if resolved.observed_flow is not None and not resolved.observed_flow.empty:
    print(f"resolved observed_flow: {resolved.observed_flow.shape}")
    print(f"columns: {list(resolved.observed_flow.columns)}")
    display(resolved.observed_flow.head(5))

if resolved.observed_inventory is not None and not resolved.observed_inventory.empty:
    print(f"resolved observed_inventory: {resolved.observed_inventory.shape}")
    print(f"columns: {list(resolved.observed_inventory.columns)}")
    display(resolved.observed_inventory.head(5))

resolved observed_flow: (548, 5)
columns: ['source_id', 'target_id', 'commodity_category', 'period_id', 'quantity']


,source_id,target_id,commodity_category,period_id,quantity
0,station_1,station_10,working_bike,p0,4.0
1,station_1,station_10,working_bike,p1,4.0
2,station_1,station_10,working_bike,p2,3.0
3,station_1,station_10,working_bike,p3,1.0
4,station_1,station_10,working_bike,p4,1.0


resolved observed_inventory: (70, 4)
columns: ['facility_id', 'commodity_category', 'period_id', 'quantity']


,facility_id,commodity_category,period_id,quantity
0,station_1,working_bike,p0,13.291667
1,station_1,working_bike,p1,15.750000
2,station_1,working_bike,p2,17.416667
3,station_1,working_bike,p3,17.458333
4,station_1,working_bike,p4,14.083333


In [7]:
print("Summary")
print(f"  trips in source:          {len(loader.source.df_trips)}")
print(f"  observed_flow rows (raw):  {len(raw.observed_flow)}")
print(f"  observed_inventory (raw):  {len(raw.observed_inventory)}")
print(f"  demand rows (raw):         {len(raw.demand)}")
if resolved.observed_flow is not None:
    print(f"  observed_flow (resolved):  {len(resolved.observed_flow)}")
if resolved.observed_inventory is not None:
    print(f"  observed_inventory (res):  {len(resolved.observed_inventory)}")

Summary
  trips in source:          1167
  observed_flow rows (raw):  548
  observed_inventory (raw):  70
  demand rows (raw):         70
  observed_flow (resolved):  548
  observed_inventory (res):  70
